# Celestack Workflow

This is a development workflow for the `celestack` project.

Before release, it will be fully substituted by a CLI, TUI, or GUI.

## Imports and Project settings

* Light Frames: 117
* Dark Frames: 32

In [1]:
from pathlib import Path

from celestack.frame import LightFrame, DarkFrame, MasterDark, AverageFrame, Mask
from celestack import progress_bar as PROGRESS_BAR

PROJECT = "test"

LF_PATHS = list(Path("/home/martin/Desktop/tenerife/lf").glob("*.tif"))
LF_NAMES = sorted(pth.stem for pth in LF_PATHS)
DF_PATHS = list(Path("/home/martin/Desktop/tenerife/df").glob("*.tif"))
DF_NAMES = sorted(pth.stem for pth in DF_PATHS)

## Initialize the frames

In [ ]:
for pth in PROGRESS_BAR(LF_PATHS, description="Loading light frames"):
    _ = LightFrame(project=PROJECT, name=pth.stem, img_path=pth)
for pth in PROGRESS_BAR(DF_PATHS, description="Loading dark frames"):
    _ = DarkFrame(project=PROJECT, name=pth.stem, img_path=pth)

## Create the Master Dark

In [ ]:
dark_frames = [
    DarkFrame.from_state(project=PROJECT, name=pth.stem)
    for pth in DF_PATHS
]

_ = MasterDark(PROJECT, "MasterDark", dark_frames)

## Subtract the Master Dark from all the light frames

In [ ]:
master_dark = MasterDark.from_state(PROJECT, "MasterDark")
for light_frame in PROGRESS_BAR(
        [LightFrame.from_state(PROJECT, name) for name in LF_NAMES],
        description="Subtracting master dark",
    ):
    light_frame.subtract_master_dark(master_dark)

## Create the foreground mask

- Create an averaged light frame.
- Determine the number of clusters (trial & error).
- Run K-Means on the full image (with the color channels and coordinates as features).
- Determine the clusters which make the foreground and use them for the mask.
- If applicable, explicitly mark some rectangles as either sky or foreground.

In [ ]:
# Create the median light frame for the mask:
_ = AverageFrame(
    project=PROJECT,
    name="MedianLight",
    frames=[LightFrame.from_state(PROJECT, name) for name in LF_NAMES],
    avg_func="median",
)

Run the K-Means custering:

In [ ]:
import plotly.graph_objects as go
from sklearn.cluster import KMeans
import numpy as np

N_CLUSTERS = 2  # number of clusters

median_light = AverageFrame.from_state(PROJECT, "MedianLight")
bit_depth = median_light.bit_depth
array = median_light.image_array

h, w, c = array.shape

# Treat the RGB channels & pixel coordinates as features:
features = array.reshape(-1, c)  # array of RGB values
features = features.astype(np.float32)
# Add the pixel coordinates:
x_coords = np.tile(np.arange(w), h).astype(np.float32)
y_coords = np.repeat(np.arange(h), w).astype(np.float32)
features = np.column_stack((features, x_coords, y_coords))
# Normalize the features:
for i in range(features.shape[1]):
    features[:, i] -= np.mean(features[:, i])
    features[:, i] /= np.std(features[:, i])

# Apply K-Means clustering
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42)
labels = kmeans.fit_predict(features)

# Relabel the clusters so they are in order of their mean values:
cluster_means = np.mean(kmeans.cluster_centers_, axis=1)
sorted_indices = np.argsort(cluster_means)
new_labels = np.zeros_like(labels)
for new_label, old_label in enumerate(sorted_indices):
    new_labels[labels == old_label] = new_label
labels = new_labels

# Create the mask as a boolean array - Cluster 0 is the foreground:
foreground_cluster_labels = [0]
mask = ~np.isin(labels, foreground_cluster_labels).reshape(h, w)

# Plot the clusters in a histogram:
fig = go.Figure()
flat_array_grayscale = median_light.compressed_array.reshape(-1)
for label in sorted(np.unique(labels)):
    cluster_pixels = flat_array_grayscale[labels == label]
    fig.add_trace(go.Histogram(
        x=cluster_pixels,
        name=f'Cluster {label}',
        opacity=0.75,
        histnorm='probability density',
    ))
fig.update_layout(
    title='Histogram of Clusters',
    xaxis_title='Pixel Value',
    yaxis_title='Density',
    barmode='overlay',
)
fig.show()

Explicitly add/remove some rectangular arrays from the mask, to get rid of the noise
in obviously sky/foreground areas

In [ ]:
mask[:1760, :] = True  # Top part of the image is sky
mask[2035:, :] = False  # Bottom part of the image is foreground

fig = go.Figure(
    data=go.Heatmap(z=mask.astype(np.uint8), colorscale='gray', showscale=False)
)
fig.update_layout(
    title="Mask",
    yaxis=dict(autorange="reversed", showgrid=False, zeroline=False),
    xaxis=dict(showgrid=False, zeroline=False),
    paper_bgcolor='rgba(255,255,255,1)',
    plot_bgcolor='rgba(255,255,255,1)'
)
fig.show()

Save the mask to the project

In [ ]:
_ = Mask(PROJECT, "Mask", img_array=mask)

## Assign the mask to all the light frames

In [ ]:
mask = Mask.from_state(PROJECT, "Mask")

for lf_name in LF_NAMES:
    light_frame = LightFrame.from_state(PROJECT, lf_name)
    light_frame.assign_mask(mask)

In [ ]:
# Show the mask has been added peristently:
light_frame = LightFrame.from_state(PROJECT, LF_NAMES[0])
light_frame.mask.plot().show()

## Segmentation of the sky

In [ ]:
raise NotImplementedError("Segmentation is not implemented yet")

## Query a segment

In [24]:
light_frame = LightFrame.from_state(PROJECT, LF_NAMES[0])
fig = light_frame.plot_segment((0, 0, 500, 500))
fig.show()